# Download the NYC taxi datasets

In [0]:
%pip install geopandas fsspec --quiet

In [0]:
import os
import datetime as dt
import zipfile
import requests
import geopandas
import fsspec

# Define the volume path
volume_path = "/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/"

# Create the directory if it doesn't exist
os.makedirs(volume_path, exist_ok=True)

# Define the URLs
urls = [
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"]

for y in range(2025, dt.datetime.now().year + 1):
    for m in range(1, 13):
        urls.append(f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet")

# Download each file
for url in urls:
    filename = url.split("/")[-1]
    filepath = os.path.join(volume_path, filename)
    
    print(f"Downloading {filename}...")
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"Successfully downloaded {filename} to {filepath}")
    else:
        print(f"Failed to download {filename}. Status code: {response.status_code}")

print("\nDownload complete!")

In [0]:
# Uncompress the zip file 
with zipfile.ZipFile("/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/taxi_zones.zip", "r") as zip_ref:
    zip_ref.extractall(volume_path)

# Ingest data to raw layer

## Ingest yellow_tripdata 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType, IntegerType, TimestampNTZType
from pyspark.sql import functions as F

# Define explicit schema for yellow taxi data
schema = StructType([
    StructField('VendorID', IntegerType(), True), 
    StructField('tpep_pickup_datetime', TimestampNTZType(), True), 
    StructField('tpep_dropoff_datetime', TimestampNTZType(), True), 
    StructField('passenger_count', LongType(), True), 
    StructField('trip_distance', DoubleType(), True), 
    StructField('RatecodeID', LongType(), True), 
    StructField('store_and_fwd_flag', StringType(), True), 
    StructField('PULocationID', IntegerType(), True), 
    StructField('DOLocationID', IntegerType(), True), 
    StructField('payment_type', LongType(), True), 
    StructField('fare_amount', DoubleType(), True), 
    StructField('extra', DoubleType(), True), 
    StructField('mta_tax', DoubleType(), True), 
    StructField('tip_amount', DoubleType(), True), 
    StructField('tolls_amount', DoubleType(), True), 
    StructField('improvement_surcharge', DoubleType(), True), 
    StructField('total_amount', DoubleType(), True), 
    StructField('congestion_surcharge', DoubleType(), True), 
    StructField('Airport_fee', DoubleType(), True), 
    StructField('cbd_congestion_fee', DoubleType(), True), 
])

# Source path
source_path = "/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/yellow_tripdata_2026-*.parquet"

# Read parquet files with explicit schema
df = spark.read \
    .schema(schema) \
    .format("parquet") \
    .load(source_path) \
    .withColumn("metadata_file_path", F.col("_metadata.file_path")) \
    .withColumn("metadata_file_name", F.col("_metadata.file_name")) \
    .withColumn("metadata_file_size", F.col("_metadata.file_size")) \
    .withColumn("metadata_file_modification_time", F.col("_metadata.file_modification_time")) \
    .withColumn("ingestion_timestamp", F.current_timestamp())    

# Write to Delta table
target_table = "dbx_joshdevph_dev.raw.stg_yellow_tripdata"
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)

In [0]:
%sql
DROP TABLE IF EXISTS dbx_joshdevph_dev.raw.stg_yellow_tripdata;

## Ingest taxi_zone_lookup

In [0]:
%sql
DROP TABLE IF EXISTS dbx_joshdevph_dev.raw.stg_taxi_zone_lookup;

In [0]:
# Source path
source_path = "/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/taxi_zone_lookup.csv"

schema = StructType([
    StructField('LocationID', StringType(), True), 
    StructField('Borough', StringType(), True), 
    StructField('Zone', StringType(), True), 
    StructField('service_zone', StringType(), True)
])

# Read csv files with explicit schema
df = spark.read \
    .schema(schema) \
    .format("csv") \
    .option("header", "true") \
    .load(source_path) \
    .withColumn("metadata_file_path", F.col("_metadata.file_path")) \
    .withColumn("metadata_file_name", F.col("_metadata.file_name")) \
    .withColumn("metadata_file_size", F.col("_metadata.file_size")) \
    .withColumn("metadata_file_modification_time", F.col("_metadata.file_modification_time")) \
    .withColumn("ingestion_timestamp", F.current_timestamp()) 

# Write to Delta table
target_table = "dbx_joshdevph_dev.raw.stg_taxi_zone_lookup"
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)

## Ingest taxi_zone shape file

In [0]:
%sql
DROP TABLE IF EXISTS dbx_joshdevph_dev.raw.stg_taxi_zone_shape;

In [0]:
gdf = geopandas.read_file("/Volumes/dbx_joshdevph_dev/raw/vlm/tlc_trip/taxi_zones/taxi_zones.shp")
gdf = gdf.to_crs(epsg=4326) # Convert it to WGS84
gdf["geometry_wkt"] = gdf.geometry.to_wkt() # Convert the geometry to WKT
gdf.columns = gdf.columns.str.lower() # Convert column names to lowercase
gdf_spark = gdf.drop(columns=["geometry"])

In [0]:
zones_df = spark.createDataFrame(gdf_spark)

# Write to Delta table
target_table = "dbx_joshdevph_dev.raw.stg_taxi_zone_shape"
zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "false") \
    .saveAsTable(target_table)

# Data quality checks 

In [0]:
%sql
CREATE OR REPLACE VIEW dbx_joshdevph_dev.processed.vw_yellow_tripdata_dq
AS

WITH dq_checks AS (

    SELECT
        *,

        -- ============================================================
        -- DQ001: Completeness
        -- Pickup datetime must not be NULL
        -- ============================================================
        CASE
            WHEN tpep_pickup_datetime IS NOT NULL
            THEN TRUE
            ELSE FALSE
        END AS dq001_pickup_datetime_not_null,


        -- ============================================================
        -- DQ002: Completeness
        -- Dropoff datetime must not be NULL
        -- ============================================================
        CASE
            WHEN tpep_dropoff_datetime IS NOT NULL
            THEN TRUE
            ELSE FALSE
        END AS dq002_dropoff_datetime_not_null,


        -- ============================================================
        -- DQ003: Consistency
        -- Dropoff datetime must not be earlier than pickup datetime
        -- ============================================================
        CASE
            WHEN tpep_pickup_datetime IS NOT NULL
             AND tpep_dropoff_datetime IS NOT NULL
             AND tpep_dropoff_datetime >= tpep_pickup_datetime
            THEN TRUE
            ELSE FALSE
        END AS dq003_valid_trip_datetime,


        -- ============================================================
        -- DQ004: Reasonability
        -- Trip distance must not be negative
        -- ============================================================
        CASE
            WHEN trip_distance IS NOT NULL
             AND trip_distance >= 0
            THEN TRUE
            ELSE FALSE
        END AS dq004_trip_distance_non_negative,


        -- ============================================================
        -- DQ005: Reasonability
        -- Passenger count must be between 0 and 8
        -- ============================================================
        CASE
            WHEN passenger_count IS NOT NULL
             AND passenger_count BETWEEN 0 AND 8
            THEN TRUE
            ELSE FALSE
        END AS dq005_passenger_count_valid,


        -- ============================================================
        -- DQ006: Reasonability
        -- Fare amount must not be negative
        -- ============================================================
        CASE
            WHEN fare_amount IS NOT NULL
             AND fare_amount >= 0
            THEN TRUE
            ELSE FALSE
        END AS dq006_fare_amount_non_negative,


        -- ============================================================
        -- DQ007: Validity
        -- Payment type must contain a recognized code
        -- ============================================================
        CASE
            WHEN payment_type IS NOT NULL
             AND payment_type IN (0, 1, 2, 3, 4, 5, 6)
            THEN TRUE
            ELSE FALSE
        END AS dq007_payment_type_valid,


        -- ============================================================
        -- DQ008: Consistency
        -- Total amount should reconcile with its components
        -- Allow a tolerance of $0.01
        -- ============================================================
        CASE
            WHEN total_amount IS NOT NULL
             AND ABS(
                    total_amount -
                    (
                        COALESCE(fare_amount, 0)
                        + COALESCE(extra, 0)
                        + COALESCE(mta_tax, 0)
                        + COALESCE(tip_amount, 0)
                        + COALESCE(tolls_amount, 0)
                        + COALESCE(improvement_surcharge, 0)
                        + COALESCE(congestion_surcharge, 0)
                        + COALESCE(Airport_fee, 0)
                        + COALESCE(cbd_congestion_fee, 0)
                    )
                 ) <= 0.01
            THEN TRUE
            ELSE FALSE
        END AS dq008_total_amount_reconciles

    FROM dbx_joshdevph_dev.raw.stg_yellow_tripdata
),

dq_results AS (

    SELECT
        *,

        -- ============================================================
        -- Overall Data Quality Status
        -- ============================================================
        CASE
            WHEN dq001_pickup_datetime_not_null
             AND dq002_dropoff_datetime_not_null
             AND dq003_valid_trip_datetime
             AND dq004_trip_distance_non_negative
             AND dq005_passenger_count_valid
             AND dq006_fare_amount_non_negative
             AND dq007_payment_type_valid
             AND dq008_total_amount_reconciles
            THEN 'PASS'
            ELSE 'FAIL'
        END AS dq_status,


        -- ============================================================
        -- Number of failed DQ rules
        -- ============================================================
        (
            CASE WHEN NOT dq001_pickup_datetime_not_null THEN 1 ELSE 0 END +
            CASE WHEN NOT dq002_dropoff_datetime_not_null THEN 1 ELSE 0 END +
            CASE WHEN NOT dq003_valid_trip_datetime THEN 1 ELSE 0 END +
            CASE WHEN NOT dq004_trip_distance_non_negative THEN 1 ELSE 0 END +
            CASE WHEN NOT dq005_passenger_count_valid THEN 1 ELSE 0 END +
            CASE WHEN NOT dq006_fare_amount_non_negative THEN 1 ELSE 0 END +
            CASE WHEN NOT dq007_payment_type_valid THEN 1 ELSE 0 END +
            CASE WHEN NOT dq008_total_amount_reconciles THEN 1 ELSE 0 END
        ) AS dq_failed_rule_count,


        -- ============================================================
        -- List of failed DQ rules
        -- ============================================================
        ARRAY_COMPACT(
            ARRAY(
                CASE
                    WHEN NOT dq001_pickup_datetime_not_null
                    THEN 'DQ001'
                END,

                CASE
                    WHEN NOT dq002_dropoff_datetime_not_null
                    THEN 'DQ002'
                END,

                CASE
                    WHEN NOT dq003_valid_trip_datetime
                    THEN 'DQ003'
                END,

                CASE
                    WHEN NOT dq004_trip_distance_non_negative
                    THEN 'DQ004'
                END,

                CASE
                    WHEN NOT dq005_passenger_count_valid
                    THEN 'DQ005'
                END,

                CASE
                    WHEN NOT dq006_fare_amount_non_negative
                    THEN 'DQ006'
                END,

                CASE
                    WHEN NOT dq007_payment_type_valid
                    THEN 'DQ007'
                END,

                CASE
                    WHEN NOT dq008_total_amount_reconciles
                    THEN 'DQ008'
                END
            )
        ) AS dq_failed_rules

    FROM dq_checks
)

SELECT *
FROM dq_results
;

In [0]:
%sql
CREATE OR REPLACE VIEW dbx_joshdevph_dev.processed.vw_yellow_tripdata_valid
AS

SELECT * EXCEPT (
    dq001_pickup_datetime_not_null,
    dq002_dropoff_datetime_not_null,
    dq003_valid_trip_datetime,
    dq004_trip_distance_non_negative,
    dq005_passenger_count_valid,
    dq006_fare_amount_non_negative,
    dq007_payment_type_valid,
    dq008_total_amount_reconciles,
    dq_status,
    dq_failed_rule_count,
    dq_failed_rules
)
FROM dbx_joshdevph_dev.processed.vw_yellow_tripdata_dq
WHERE dq_status = 'PASS';

In [0]:
%sql
CREATE OR REPLACE VIEW dbx_joshdevph_dev.processed.vw_yellow_tripdata_quarantine
AS
SELECT *
FROM dbx_joshdevph_dev.processed.vw_yellow_tripdata_dq
WHERE dq_status = 'FAIL';

# Medallion

In [0]:
%sql
CREATE OR REPLACE TABLE dbx_joshdevph_dev.mart.dim_location
USING DELTA
AS
select 
    tzl.LocationID AS location_id
    ,tzl.borough AS borough
    ,tzl.zone
    ,tzl.service_zone
    ,tzs.shape_leng
    ,tzs.shape_area
    ,tzs.geometry_wkt
from dbx_joshdevph_dev.raw.stg_taxi_zone_lookup tzl
left join dbx_joshdevph_dev.raw.stg_taxi_zone_shape tzs on tzl.locationid = tzs.locationid

In [0]:
%sql
CREATE OR REPLACE TABLE dbx_joshdevph_dev.mart.fact_daily_yellow_taxi_trips
USING DELTA
AS

SELECT
    -- ============================================================
    -- Dimension Keys
    -- ============================================================
    CAST(DATE_FORMAT(tpep_pickup_datetime, 'yyyyMMdd') AS INT)
        AS date_key,

    PULocationID,
    DOLocationID,

    -- ============================================================
    -- Trip Measures
    -- ============================================================
    COUNT(*) AS total_trips,
    SUM(passenger_count) AS total_passenger_count,
    AVG(passenger_count) AS avg_passenger_count,
    SUM(trip_distance) AS total_trip_distance,
    AVG(trip_distance) AS avg_trip_distance,
    AVG(
        TIMESTAMPDIFF(
            MINUTE,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        )
    ) AS avg_trip_duration_minutes,

    -- ============================================================
    -- Fare Measures
    -- ============================================================
    SUM(fare_amount) AS total_fare_amount,
    AVG(fare_amount) AS avg_fare_amount,

    -- ============================================================
    -- Tip Measures
    -- ============================================================
    SUM(tip_amount) AS total_tip_amount,
    AVG(tip_amount) AS avg_tip_amount,

    -- ============================================================
    -- Toll Measures
    -- ============================================================
    SUM(tolls_amount) AS total_tolls_amount,
    AVG(tolls_amount) AS avg_tolls_amount,

    -- ============================================================
    -- Tax Measures
    -- ============================================================
    SUM(mta_tax) AS total_mta_tax,
    AVG(mta_tax) AS avg_mta_tax,

    -- ============================================================
    -- Other Fees / Surcharges
    -- ============================================================
    SUM(extra) AS total_extra,
    AVG(extra) AS avg_extra,
    SUM(improvement_surcharge) AS total_improvement_surcharge,
    AVG(improvement_surcharge) AS avg_improvement_surcharge,
    SUM(congestion_surcharge) AS total_congestion_surcharge,
    AVG(congestion_surcharge) AS avg_congestion_surcharge,
    SUM(Airport_fee) AS total_airport_fee,
    AVG(Airport_fee) AS avg_airport_fee,
    SUM(cbd_congestion_fee) AS total_cbd_congestion_fee,
    AVG(cbd_congestion_fee) AS avg_cbd_congestion_fee,

    -- ============================================================
    -- Overall Amount Measures
    -- ============================================================
    SUM(total_amount) AS total_amount,
    AVG(total_amount) AS avg_total_amount,

    -- ============================================================
    -- Derived Measures
    -- ============================================================
    AVG(
        CASE
            WHEN fare_amount > 0
            THEN (tip_amount / fare_amount) * 100
        END
    ) AS avg_tip_percentage,
    AVG(
        CASE
            WHEN trip_distance > 0
            THEN fare_amount / trip_distance
        END
    ) AS avg_fare_per_mile,
    AVG(
        CASE
            WHEN passenger_count > 0
            THEN total_amount / passenger_count
        END
    ) AS avg_amount_per_passenger

FROM dbx_joshdevph_dev.processed.vw_yellow_tripdata_valid

GROUP BY
    CAST(DATE_FORMAT(tpep_pickup_datetime, 'yyyyMMdd') AS INT),
    PULocationID,
    DOLocationID;

In [0]:
%sql

CREATE OR REPLACE VIEW dbx_joshdevph_dev.mart.vw_yellow_taxi_daily
AS

SELECT

    -- ============================================================
    -- Date
    -- ============================================================
    f.date_key,


    -- ============================================================
    -- Pickup Location
    -- ============================================================
    f.PULocationID AS pickup_location_id,
    pu.borough AS pickup_borough,
    pu.zone AS pickup_zone,
    pu.service_zone AS pickup_service_zone,


    -- ============================================================
    -- Dropoff Location
    -- ============================================================
    f.DOLocationID AS dropoff_location_id,
    do.borough AS dropoff_borough,
    do.zone AS dropoff_zone,
    do.service_zone AS dropoff_service_zone,


    -- ============================================================
    -- Trip Measures
    -- ============================================================
    f.total_trips,
    f.total_passenger_count,
    f.avg_passenger_count,
    f.total_trip_distance,
    f.avg_trip_distance,
    f.avg_trip_duration_minutes,


    -- ============================================================
    -- Fare Measures
    -- ============================================================
    f.total_fare_amount,
    f.avg_fare_amount,


    -- ============================================================
    -- Tip Measures
    -- ============================================================
    f.total_tip_amount,
    f.avg_tip_amount,


    -- ============================================================
    -- Toll Measures
    -- ============================================================
    f.total_tolls_amount,
    f.avg_tolls_amount,


    -- ============================================================
    -- Tax Measures
    -- ============================================================
    f.total_mta_tax,
    f.avg_mta_tax,


    -- ============================================================
    -- Fees and Surcharges
    -- ============================================================
    f.total_extra,
    f.avg_extra,

    f.total_improvement_surcharge,
    f.avg_improvement_surcharge,

    f.total_congestion_surcharge,
    f.avg_congestion_surcharge,

    f.total_airport_fee,
    f.avg_airport_fee,

    f.total_cbd_congestion_fee,
    f.avg_cbd_congestion_fee,


    -- ============================================================
    -- Overall Amount Measures
    -- ============================================================
    f.total_amount,
    f.avg_total_amount,


    -- ============================================================
    -- Derived Measures
    -- ============================================================
    f.avg_tip_percentage,
    f.avg_fare_per_mile,
    f.avg_amount_per_passenger


FROM dbx_joshdevph_dev.mart.fact_daily_yellow_taxi_trips f

LEFT JOIN dbx_joshdevph_dev.mart.dim_location pu
    ON f.PULocationID = pu.location_id

LEFT JOIN dbx_joshdevph_dev.mart.dim_location do
    ON f.DOLocationID = do.location_id;